# 02 — Randomization Methods in Online Experiments
**Prerequisites:** `01_ab_testing_fundamentals.ipynb`; causal_inference_course/02_randomized_experiments.ipynb
(stratified randomization theory).
**Connects to:** `06_sample_ratio_mismatch.ipynb` (balance checks after the fact);
experimental_design_course/02_blocking_designs.ipynb (blocking theory).

## Narrative thread
```
Unit of randomization -> hash-based deterministic bucketing -> stratified randomization
   -> balance checks -> common bugs
```

## Unit of randomization

The unit you randomize must match the unit at which interference is negligible and the unit the
business decision is made about:

| Unit | When to use | Risk if wrong |
|---|---|---|
| User (logged-in ID) | Most product features | Users switching devices dilute the effect if you randomize by device/cookie instead |
| Session | Ephemeral UI experiments (e.g. search ranking) | A user can see both arms across sessions -> contamination |
| Cluster (market, region, time-slot) | Two-sided marketplaces, pricing, feed ranking with liquidity effects | Coarser unit -> fewer independent units -> less power (see experimental_design_course/08) |

## Hash-based deterministic bucketing

Real experimentation platforms almost never store a per-user assignment table. Instead they
compute a **deterministic hash** of the user id and the experiment's salt, so that:

- the same user always lands in the same arm for the *same* experiment (consistency across
  sessions/devices, as long as the id is stable),
- two *different* experiments that use different salts assign essentially independently (so a
  user can be in treatment for experiment A and control for experiment B without correlation),
- no database lookup is needed at request time — the assignment is a pure function.

$$\text{bucket}(u) = \text{hash}(u \Vert \text{salt}) \bmod 100$$

and then a contiguous range of buckets (e.g. `[0, 50)` vs `[50, 100)`) is mapped to each arm. Using
a *good* hash function (MD5, SHA-1, or a fast 32/64-bit hash like MurmurHash) is essential: a poor
hash can produce non-uniform buckets, correlated assignment across experiments that share a salt
structure, or exploitable assignment if users can predict their bucket.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [2]:
# ── Hash-based deterministic bucketing, exactly as a real platform would do it ──
import hashlib

def assign_bucket(user_id: str, salt: str, n_buckets: int = 100) -> int:
    h = hashlib.sha256(f"{user_id}-{salt}".encode()).hexdigest()
    return int(h, 16) % n_buckets

def assign_arm(user_id: str, salt: str, treatment_frac: float = 0.5) -> str:
    bucket = assign_bucket(user_id, salt)
    return 'treatment' if bucket < treatment_frac * 100 else 'control'

user_ids = [f"user_{i}" for i in range(20000)]
salt_a = "checkout_redesign_v1"
salt_b = "search_ranking_v3"     # a completely different experiment

arms_a = pd.Series([assign_arm(u, salt_a) for u in user_ids])
arms_b = pd.Series([assign_arm(u, salt_b) for u in user_ids])

print("Experiment A allocation:\n", arms_a.value_counts(normalize=True))
print("\nExperiment B allocation:\n", arms_b.value_counts(normalize=True))

# Consistency check: the same user always gets the same arm within an experiment
repeat_check = all(assign_arm(u, salt_a) == arms_a[i] for i, u in enumerate(user_ids[:500]))
print("\nDeterministic / repeatable assignment for experiment A:", repeat_check)

# Independence check: cross-tab of arm in A vs arm in B should be ~independent (no shared salt structure)
crosstab = pd.crosstab(arms_a, arms_b, normalize=True)
print("\nJoint allocation across two unrelated experiments (should be ~25% each cell):\n", crosstab)

Experiment A allocation:
 control      0.50155
treatment    0.49845
Name: proportion, dtype: float64

Experiment B allocation:
 control      0.50215
treatment    0.49785
Name: proportion, dtype: float64

Deterministic / repeatable assignment for experiment A: True

Joint allocation across two unrelated experiments (should be ~25% each cell):
 col_0      control  treatment
row_0                        
control    0.24940    0.25215
treatment  0.25275    0.24570


## Stratified randomization

When a covariate is known in advance to be prognostic (e.g. country, device platform, historical
spend tier), randomizing **within strata** (then combining with the stratified estimator from
`causal_inference_course/02_randomized_experiments.ipynb`) guarantees balance on it and can
improve power, especially at moderate sample sizes. At the scale of most online experiments
(hundreds of thousands to millions of users), simple hash-based randomization already balances
covariates well by the law of large numbers, so stratification is used mainly for:

- experiments with limited sample size (new market launches, small segments),
- guaranteeing exact balance on a covariate that will be a **primary segment cut** later
  (see `09_segmentation_heterogeneity.ipynb`).

## Balance checks: does the randomization look right?

Even with a correct hash function, always verify empirically that pre-experiment covariates and,
critically, the **allocation ratio itself**, look balanced. A ratio that deviates from the
intended split (e.g. 50.4% vs 49.6% turning out to be 52% vs 48% at n = 2,000,000) is a
**sample ratio mismatch (SRM)** — a red flag that something in the pipeline (not the randomizer)
is broken. SRM detection via a chi-square goodness-of-fit test is covered in full in
`06_sample_ratio_mismatch.ipynb`; here we just confirm the *randomizer itself* produces a clean
50/50 split before any pipeline logic (bot filters, redirects, etc.) touches it.


In [3]:
# ── Balance preview: chi-square check straight off the randomizer (before any pipeline logic) ──
obs_counts = arms_a.value_counts().sort_index().values   # [control, treatment]
expected = np.array([len(arms_a) / 2, len(arms_a) / 2])
chi2, p_value = stats.chisquare(obs_counts, expected)
print(f"Observed counts: {obs_counts}, expected: {expected}")
print(f"Chi-square = {chi2:.4f}, p = {p_value:.4f}  -> clean randomizer should NOT reject at typical alpha")
print("A full SRM investigation (with real pipeline data) is done in 06_sample_ratio_mismatch.ipynb")

Observed counts: [10031  9969], expected: [10000. 10000.]
Chi-square = 0.1922, p = 0.6611  -> clean randomizer should NOT reject at typical alpha
A full SRM investigation (with real pipeline data) is done in 06_sample_ratio_mismatch.ipynb


## Common bugs in randomization

1. **Non-uniform hashing.** Using a weak or poorly-seeded hash (e.g. Python's built-in `hash()`,
   which is randomized per process by default and not a real cryptographic/statistical hash) can
   produce visibly uneven buckets or, worse, buckets that are *not stable across requests*.
2. **Salt reuse ("correlated experiments").** If two unrelated experiments reuse the same salt (or
   derive their salt from the same seed incorrectly), a user's bucket becomes correlated across
   experiments, and interaction effects get silently confounded with main effects.
3. **Re-randomization / re-bucketing** when the same experiment is redeployed with a new build can
   flip users between arms mid-experiment, contaminating both arms — the experiment ID/salt must
   stay fixed for the life of the experiment.
4. **Post-randomization filtering** (e.g., excluding "bot-like" users differently by arm, or
   redirect logic that fires only for one arm) reintroduces selection bias and is a classic SRM
   root cause — see `06_sample_ratio_mismatch.ipynb`.

## Key takeaways

| Concept | Statement |
|---|---|
| Unit of randomization | Match it to the unit where interference is negligible & the decision is made |
| Hash bucketing | `hash(user_id + salt) % 100`; deterministic, stateless, independent across experiments with distinct salts |
| Stratification | Guarantees balance in small samples; usually unnecessary at typical online-experiment scale |
| Balance checks | Chi-square on allocation counts even on the raw randomizer output; full SRM tooling in notebook 06 |

## References

- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*, Ch. 3 (experimentation platform architecture) & Ch. 15 (implementation).
- Xie, H., & Aurisset, J. (2016). Diagnosing Sample Ratio Mismatch in Online Controlled Experiments. *KDD 2016*.
- See `causal_inference_course/02_randomized_experiments.ipynb` for the stratified estimator's formal variance.
